# Áp dụng & Triển khai Môi trường Baseline FinRL cho Bài toán Portfolio Management (Tối ưu hóa danh mục)

---
## 1. Cài đặt và Nạp Thư viện Cần thiết


In [1]:
# Cài đặt các thư viện phụ thuộc nếu chạy trên môi trường mới (Colab / Kaggle)
# !pip install gymnasium stable-baselines3 sb3_contrib torch numpy pandas matplotlib yfinance scikit-learn quantstats

import os
import sys
import math
import types
import tempfile
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

# Cấu hình MPLCONFIGDIR tránh cảnh báo Matplotlib cache trên Windows
if 'MPLCONFIGDIR' not in os.environ:
    os.environ['MPLCONFIGDIR'] = tempfile.mkdtemp()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

warnings.filterwarnings('ignore')

# Cấu hình seed cho tính tái lập thực nghiệm (Reproducibility)
SEED = 42
np.random.seed(SEED)

# Khởi tạo Gym compatibility shim cho các module FinRL
sys.modules.setdefault('gym', gym)
sys.modules.setdefault('gym.spaces', gym.spaces)
sys.modules.setdefault('gym.utils', gym.utils)
if hasattr(gym, 'utils') and hasattr(gym.utils, 'seeding'):
    sys.modules.setdefault('gym.utils.seeding', gym.utils.seeding)

# Khởi tạo QuantStats mock fallback nếu chưa cài đặt quantstats để môi trường FinRL chạy trơn tru
try:
    import quantstats as qs
except ImportError:
    qs = types.ModuleType('quantstats')
    qs.stats = types.SimpleNamespace(
        max_drawdown=lambda x: float(np.min(x) / np.max(x) - 1.0) if len(x) > 0 else 0.0,
        sharpe=lambda x: float(np.mean(x) / (np.std(x) + 1e-8) * np.sqrt(252)) if len(x) > 0 else 0.0,
    )
    qs.plots = types.SimpleNamespace(snapshot=lambda *args, **kwargs: None)
    sys.modules['quantstats'] = qs

# Đảm bảo đường dẫn mã nguồn FinRL được nạp vào sys.path
finrl_path = os.path.abspath(os.path.join(os.getcwd(), 'FinRL-master'))
if finrl_path not in sys.path:
    sys.path.insert(0, finrl_path)

# Nạp trực tiếp lớp môi trường PortfolioOptimizationEnv chính thức của FinRL (Costa & Costa, 2023)
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv

print('✅ Đã nạp thành công các thư viện cốt lõi và môi trường PortfolioOptimizationEnv chính thức của FinRL!')


✅ Đã nạp thành công các thư viện cốt lõi và môi trường PortfolioOptimizationEnv chính thức của FinRL!


---
## 2. Cơ sở Toán học & Thiết kế Môi trường FinRL Portfolio Optimization

### Tham chiếu Bài báo Khoa học Cốt lõi:
- **Caio Costa & Anna Costa (2023)**: *"POE: A General Portfolio Optimization Environment for FinRL"*, Anais do II Brazilian Workshop on Artificial Intelligence in Finance (BWAIF), pp. 132–143. https://doi.org/10.5753/bwaif.2023.231144.
- **Zhengyao Jiang, Dixing Xu, & Jinjun Liang (2017)**: *"A Deep Reinforcement Learning Framework for the Financial Portfolio Management Problem"*, arXiv:1706.10059.

### Các phương trình cốt lõi trong Môi trường:
1. **Vector biến động giá tương đối (Relative Price Vector)** tại bước $t$:
   $$\mathbf{y}_t = \left[1.0, \frac{p_{t, 1}}{p_{t-1, 1}}, \frac{p_{t, 2}}{p_{t-1, 2}}, \dots, \frac{p_{t, N}}{p_{t-1, N}}\right]^\top \in \mathbb{R}^{N+1}$$
   *Index 0 là Tiền mặt (Cash), luôn có $y_t^{(0)} = 1.0$ (không chịu rủi ro biến động giá cổ phiếu)*.

2. **Ràng buộc Simplex cho Vector Trọng số (Action)**:
   $$\mathbf{w}_t \in \Delta^N \iff w_{t, i} \ge 0 \quad \text{và} \quad \sum_{i=0}^N w_{t, i} = 1$$
   Môi trường tự động chuẩn hóa vector hành động về không gian Simplex bằng Softmax normalization.

3. **Hiện tượng Trôi dạt Trọng số theo Giá thị trường (Price Drift)**:
   Trước khi tái cân bằng ở bước $t$, trọng số thực tế $\mathbf{w}_{t-1}$ bị thay đổi do giá cổ phiếu thay đổi:
   $$\mathbf{w}'_{t-1} = \frac{\mathbf{y}_t \odot \mathbf{w}_{t-1}}{\mathbf{y}_t^\top \mathbf{w}_{t-1}}$$

4. **Mô hình Phí Giao dịch (Transaction Remainder Factor - TRF)**:
   Hệ số bảo toàn vốn $\mu_t \in (0, 1]$ sau khi trừ phí giao dịch $c_{\text{fee}}$ được tính chính xác bằng thuật toán lặp điểm bất động (Jiang et al., 2017):
   $$\mu_t^{(k+1)} = \frac{1 - c_p w'_{t-1, 0} - (c_s + c_p - c_s c_p) \sum_{i=1}^N \max(w'_{t-1, i} - \mu_t^{(k)} w_{t, i}, 0)}{1 - c_p w_{t, 0}}$$

5. **Cập nhật Giá trị Danh mục & Phần thưởng (Logarithmic Return)**:
   $$V_t = \mu_t \cdot V_{t-1} \cdot (\mathbf{y}_t^\top \mathbf{w}_t)$$
   $$r_t = \ln\left(\frac{V_t}{V_{t-1}}\right) \times \text{reward\_scaling} = \left[ \ln(\mu_t) + \ln(\mathbf{y}_t^\top \mathbf{w}_t) \right] \times \text{reward\_scaling}$$


In [2]:
# ====================================================================
# ÁP DỤNG TRỰC TIẾP MÔI TRƯỜNG PortfolioOptimizationEnv CHÍNH THỨC CỦA FinRL
# (Costa & Costa 2023, BWAIF - Kế thừa trực tiếp, KHÔNG rewrite lại môi trường)
# ====================================================================

class FinRLPortfolioOptimizationEnv(PortfolioOptimizationEnv):
    """
    Kế thừa và áp dụng trực tiếp PortfolioOptimizationEnv chính thức của FinRL (Costa & Costa, 2023)
    đã được công bố và kiểm chứng trong file FinRL_PortfolioOptimizationEnv_Demo.ipynb.
    
    Môi trường mô phỏng đầy đủ:
    - Vector biến động giá tương đối y_t = [1.0, p_t / p_{t-1}]
    - Mô hình phí bảo toàn vốn TRF (Transaction Remainder Factor, Jiang et al. 2017)
    - Trôi dạt trọng số thị trường (Price Drift)
    - Phần thưởng log-return: r_t = ln(mu_t * y_t^T * w_t)
    - Không gian quan sát Tensor 3D (F x N x M) và Action Simplex (N + 1)
    """
    def __init__(
        self,
        df: Optional[pd.DataFrame] = None,
        initial_amount: float = 100_000.0,
        comission_fee_pct: float = 0.0025,
        time_window: int = 30,
        features: Optional[List[str]] = None,
        normalize_df: Optional[str] = None,
        return_last_action: bool = False,
        new_gym_api: bool = True,
        # Các tham số hỗ trợ linh hoạt cho cả DataFrame FinRL và ma trận giá
        df_features: Optional[np.ndarray] = None,
        df_prices: Optional[np.ndarray] = None,
        tic_list: Optional[List[str]] = None,
        fee_rate: Optional[float] = None,
        fee_model: str = "trf",
        reward_scaling: float = 1.0,
        **kwargs
    ):
        if features is None:
            features = ["close", "high", "low"]
        if fee_rate is not None:
            comission_fee_pct = fee_rate

        # Chuyển đổi dữ liệu sang định dạng DataFrame chuẩn FinRL nếu truyền ma trận giá
        if df is None and df_prices is not None:
            T_len, N_len = df_prices.shape
            if tic_list is None:
                tic_list = [f"TIC_{i}" for i in range(N_len)]
            date_range = pd.date_range("2020-01-01", periods=T_len, freq="B").strftime("%Y-%m-%d")
            records = []
            for t_idx, d_str in enumerate(date_range):
                for n_idx, tic in enumerate(tic_list):
                    close_p = float(df_prices[t_idx, n_idx])
                    records.append({
                        "date": d_str,
                        "tic": tic,
                        "close": close_p,
                        "high": close_p * 1.002,
                        "low": close_p * 0.998
                    })
            df = pd.DataFrame(records)

        # Gọi trực tiếp khởi tạo của PortfolioOptimizationEnv từ FinRL (KHÔNG rewrite)
        super().__init__(
            df=df,
            initial_amount=initial_amount,
            return_last_action=return_last_action,
            normalize_df=normalize_df,
            reward_scaling=reward_scaling,
            comission_fee_model=fee_model,
            comission_fee_pct=comission_fee_pct,
            features=features,
            time_window=time_window,
            new_gym_api=new_gym_api,
            **kwargs
        )

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        """Khởi tạo lại môi trường tương thích Gymnasium API mới."""
        if seed is not None:
            self._seed(seed)
        res = super().reset()
        if not self._new_gym_api and not isinstance(res, tuple):
            return res, {}
        return res

    def step(self, actions: np.ndarray):
        """Thực thi một bước mô phỏng với PortfolioOptimizationEnv chính thức."""
        res = super().step(actions)
        if len(res) == 5:
            obs, reward, terminal, truncated, info = res
        else:
            obs, reward, terminal, info = res
            truncated = False
        # Bổ sung các trường tiện ích cho tương thích đánh giá trong notebook
        info["fee_factor"] = info.get("trf_mu", 1.0)
        info["portfolio_value"] = self._portfolio_value
        return obs, reward, terminal, truncated, info

    @property
    def action_dim(self) -> int:
        return self.action_space.shape[0]

    @property
    def num_stocks(self) -> int:
        return self.portfolio_size

    @property
    def time_window(self) -> int:
        return self._time_window

    @property
    def num_features(self) -> int:
        return len(self._features)

    @property
    def T(self) -> int:
        return self.episode_length

    @property
    def asset_memory(self) -> list:
        return self._asset_memory.get("final", [self._initial_amount])

    def compute_metrics(self) -> Dict[str, float]:
        """Tính toán các chỉ số tài chính đánh giá hiệu suất danh mục."""
        values = np.array(self.asset_memory)
        if len(values) < 2:
            return {"Final Value ($)": self._portfolio_value}
        returns = np.diff(values) / values[:-1]
        total_return = (values[-1] - values[0]) / values[0]
        ann_return = ((1.0 + total_return) ** (252.0 / len(values))) - 1.0 if len(values) > 0 else 0.0
        ann_volatility = np.std(returns) * np.sqrt(252.0) if len(returns) > 1 else 0.0
        sharpe = (ann_return - 0.02) / (ann_volatility + 1e-8) if ann_volatility > 0 else 0.0
        cummax = np.maximum.accumulate(values)
        drawdowns = (cummax - values) / cummax
        max_drawdown = np.max(drawdowns) if len(drawdowns) > 0 else 0.0
        return {
            'Initial Value ($)': self._initial_amount,
            'Final Value ($)': values[-1],
            'Cumulative Return (%)': total_return * 100.0,
            'Annualized Return (%)': ann_return * 100.0,
            'Annualized Volatility (%)': ann_volatility * 100.0,
            'Sharpe Ratio': sharpe,
            'Max Drawdown (%)': max_drawdown * 100.0
        }

    def plot_results(self, title: str = "Portfolio Value Over Time"):
        """Hiển thị đồ thị tăng trưởng giá trị danh mục."""
        values = self.asset_memory
        plt.figure(figsize=(10, 5))
        plt.plot(values, label="Portfolio Value", color="#1f77b4", lw=2.0)
        plt.title(title, fontsize=12, fontweight="bold")
        plt.xlabel("Trading Steps", fontsize=10)
        plt.ylabel("Portfolio Value ($)", fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=10)
        plt.show()

print('✅ Đã áp dụng thành công lớp môi trường PortfolioOptimizationEnv chính thức của FinRL!')


✅ Đã áp dụng thành công lớp môi trường PortfolioOptimizationEnv chính thức của FinRL!


---
## 3. Pipeline Dữ liệu Thị trường & Tiền xử lý

Môi trường cần 2 đầu vào dữ liệu:
1. **Tensor Đặc trưng `df_features`** kích thước $(T \times F \times N \times M)$:
   - $T$: Số ngày giao dịch.
   - $F$: Số đặc trưng (Close, High, Low, Open).
   - $N$: Số mã cổ phiếu.
   - $M$: Kích thước cửa sổ trượt quá khứ (Time Window / Lookback).
2. **Ma trận Giá `df_prices`** kích thước $(T \times N)$ dùng để tính toán biến động giá thực tế $\mathbf{y}_t$.

Dưới đây là module tự động:
- Cho phép tải dữ liệu lịch sử thực tế từ **Yahoo Finance** nếu có kết nối Internet.
- Hoặc tự động tạo tập dữ liệu tài chính mô phỏng chân thực (**Correlated Geometric Brownian Motion - GBM**) nếu chạy offline, đảm bảo notebook **luôn luôn chạy thành công ngay lập tức**!


In [3]:
def generate_market_data(
    tickers: List[str] = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'WMT'],
    start_date: str = '2020-01-01',
    end_date: str = '2023-12-31',
    window_size: int = 30,
    features: List[str] = ['Close', 'High', 'Low', 'Open'],
    use_yfinance: bool = True
) -> Tuple[np.ndarray, np.ndarray, List[str], List[str]]:
    """
    Chuẩn bị dữ liệu lịch sử hoặc mô phỏng cho FinRLPortfolioOptimizationEnv.
    """
    N = len(tickers)
    F = len(features)
    M = window_size
    
    raw_df_dict = {}
    download_success = False
    
    if use_yfinance:
        try:
            import yfinance as yf
            print(f'>>> Đang tải dữ liệu {N} mã từ Yahoo Finance ({start_date} đến {end_date})...')
            df_raw = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False)
            
            if len(df_raw) > M + 50:
                for tic in tickers:
                    df_t = df_raw[tic][features].copy()
                    df_t = df_t.ffill().bfill()
                    raw_df_dict[tic] = df_t
                download_success = True
                print('>>> Tải dữ liệu Yahoo Finance thành công!')
        except Exception as e:
            print(f'>>> Không thể tải từ yfinance ({e}). Đang chuyển sang bộ sinh dữ liệu mô phỏng GBM...')
            download_success = False
            
    if not download_success:
        print('>>> Sinh tập dữ liệu thị trường mô phỏng (Correlated Geometric Brownian Motion)...')
        date_range = pd.date_range(start=start_date, periods=1000, freq='B')
        T_total = len(date_range)
        
        rand_mat = np.random.randn(N, N)
        corr = np.dot(rand_mat, rand_mat.T)
        d = np.diag(1.0 / np.sqrt(np.diag(corr)))
        corr = np.dot(np.dot(d, corr), d)
        
        volatilities = np.random.uniform(0.15, 0.35, N) / math.sqrt(252.0)
        cov = np.outer(volatilities, volatilities) * corr
        cholesky = np.linalg.cholesky(cov)
        
        mu = np.random.uniform(0.05, 0.20, N) / 252.0
        init_prices = np.random.uniform(50.0, 200.0, N)
        
        price_series = np.zeros((T_total, N))
        price_series[0] = init_prices
        
        for t in range(1, T_total):
            z = np.random.randn(N)
            drift = mu - 0.5 * (volatilities**2)
            diffusion = np.dot(cholesky, z)
            price_series[t] = price_series[t-1] * np.exp(drift + diffusion)
            
        for i, tic in enumerate(tickers):
            close = price_series[:, i]
            high = close * (1.0 + np.random.uniform(0.002, 0.02, T_total))
            low = close * (1.0 - np.random.uniform(0.002, 0.02, T_total))
            open_p = low + (high - low) * np.random.uniform(0.2, 0.8, T_total)
            raw_df_dict[tic] = pd.DataFrame({
                'Close': close, 'High': high, 'Low': low, 'Open': open_p
            }, index=date_range)
            
    common_dates = raw_df_dict[tickers[0]].index
    for tic in tickers[1:]:
        common_dates = common_dates.intersection(raw_df_dict[tic].index)
    common_dates = sorted(common_dates)
    
    T_target = len(common_dates) - M
    features_tensor = np.zeros((T_target, F, N, M), dtype=np.float32)
    closing_prices = np.zeros((T_target, N), dtype=np.float32)
    
    print(f'>>> Xây dựng Tensor quan sát cho {T_target} ngày giao dịch...')
    for t_idx in range(T_target):
        window_dates = common_dates[t_idx : t_idx + M]
        curr_date = common_dates[t_idx + M - 1]
        
        for n_idx, tic in enumerate(tickers):
            df_t = raw_df_dict[tic].loc[window_dates, features].values
            max_val = np.max(np.abs(df_t), axis=0, keepdims=True) + 1e-8
            norm_window = df_t / max_val
            features_tensor[t_idx, :, n_idx, :] = norm_window.T
            closing_prices[t_idx, n_idx] = raw_df_dict[tic].loc[curr_date, 'Close']
            
    print(f'✅ Tensor đặc trưng hoàn thành: {features_tensor.shape} (T x F x N x M)')
    print(f'✅ Ma trận giá hoàn thành: {closing_prices.shape} (T x N)')
    return features_tensor, closing_prices, tickers, [str(d.date()) if hasattr(d, 'date') else str(d) for d in common_dates[M:]]


---
## 4. Tạo Dữ liệu & Phân chia Tập Train - Test

Ta chọn danh mục gồm **10 cổ phiếu lớn** và phân tách dữ liệu:
- **Tập Huấn luyện (Train Set)**: 70% thời gian đầu để agent học chiến lược phân bổ.
- **Tập Kiểm thử (Test Set / Out-of-Sample)**: 30% thời gian sau để đánh giá hiệu suất thực tế mà không bị Look-Ahead Bias.


In [4]:
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'WMT']
WINDOW_SIZE = 30  # Lookback window M = 30 ngày

# Sinh dữ liệu
features_tensor, closing_prices, tickers, dates = generate_market_data(
    tickers=TICKERS,
    start_date='2020-01-01',
    end_date='2023-12-31',
    window_size=WINDOW_SIZE,
    use_yfinance=False  # Đặt True nếu bạn muốn tải trực tiếp từ Yahoo Finance khi có mạng
)

# Phân chia 70% Train - 30% Test theo chuỗi thời gian
T_total = features_tensor.shape[0]
train_size = int(T_total * 0.7)

train_features = features_tensor[:train_size]
train_prices = closing_prices[:train_size]

test_features = features_tensor[train_size:]
test_prices = closing_prices[train_size:]

print(f'Kích thước tập Train: {train_features.shape[0]} ngày giao dịch')
print(f'Kích thước tập Test:  {test_features.shape[0]} ngày giao dịch')


>>> Sinh tập dữ liệu thị trường mô phỏng (Correlated Geometric Brownian Motion)...
>>> Xây dựng Tensor quan sát cho 970 ngày giao dịch...
✅ Tensor đặc trưng hoàn thành: (970, 4, 10, 30) (T x F x N x M)
✅ Ma trận giá hoàn thành: (970, 10) (T x N)
Kích thước tập Train: 679 ngày giao dịch
Kích thước tập Test:  291 ngày giao dịch


---
## 5. Khởi tạo Môi trường Huấn luyện và Kiểm thử


In [5]:
train_env = FinRLPortfolioOptimizationEnv(
    df_features=train_features,
    df_prices=train_prices,
    tic_list=TICKERS,
    initial_amount=100_000.0,
    fee_rate=0.0025,           # Phí giao dịch 0.25%
    fee_model='trf',           # Mô hình TRF chuẩn Jiang et al.
    return_last_action=False   # Trả về trực tiếp Box tensor (F, N, M) để dễ kết nối SB3
)

test_env = FinRLPortfolioOptimizationEnv(
    df_features=test_features,
    df_prices=test_prices,
    tic_list=TICKERS,
    initial_amount=100_000.0,
    fee_rate=0.0025,
    fee_model='trf',
    return_last_action=False
)

# Kiểm tra Sanity Check
obs, info = train_env.reset()
print('✅ Khởi tạo môi trường thành công!')
print(f'- Không gian quan sát (Observation Space): {train_env.observation_space.shape}')
print(f'- Không gian hành động (Action Space):      {train_env.action_space.shape}')
print(f'- Số lượng tài sản (1 Cash + {train_env.num_stocks} Stocks): {train_env.action_dim}')

# Thử nghiệm 1 bước Step với trọng số phân bổ đều
dummy_action = np.ones(train_env.action_dim, dtype=np.float32) / train_env.action_dim
next_obs, reward, terminated, truncated, step_info = train_env.step(dummy_action)

print('\n>>> Kết quả 1 bước tái cân bằng thử nghiệm:')
print(f'  + Phí bảo toàn vốn (TRF mu_1): {step_info["fee_factor"]:.6f}')
print(f'  + Log Return (Reward r_1):     {reward:.6f}')
print(f'  + Giá trị tài sản sau bước 1:  ${step_info["portfolio_value"]:,.2f}')


✅ Khởi tạo môi trường thành công!
- Không gian quan sát (Observation Space): (3, 10, 30)
- Không gian hành động (Action Space):      (11,)
- Số lượng tài sản (1 Cash + 10 Stocks): 11

>>> Kết quả 1 bước tái cân bằng thử nghiệm:
  + Phí bảo toàn vốn (TRF mu_1): 1.000000
  + Log Return (Reward r_1):     -0.008927
  + Giá trị tài sản sau bước 1:  $99,111.29


---
## 6. Đánh giá Thực nghiệm Với Các Chiến Lược Chuẩn (Benchmarks)

Để đánh giá tính đúng đắn và hiệu quả của môi trường, ta tiến hành chạy mô phỏng 3 chiến lược chuẩn trên tập **Test**:
1. **Uniform Buy and Hold (Equal Weight - $1/N$)**: Danh mục thụ động phân bổ đều vào tất cả các tài sản. Đây là benchmark kinh điển trong lý thuyết danh mục đầu tư.
2. **Cash Only (100% Tiền mặt)**: Chiến lược phi rủi ro, nắm giữ toàn bộ vốn bằng tiền mặt.
3. **Random Rebalancing Agent**: Tác tử tái cân bằng ngẫu nhiên các trọng số qua từng phiên để quan sát ảnh hưởng của chi phí giao dịch (Turnover Cost Drag).


In [6]:
def evaluate_strategy(env: FinRLPortfolioOptimizationEnv, strategy_name: str, policy_fn) -> Dict[str, float]:
    obs, info = env.reset()
    terminated = False
    truncated = False
    
    while not (terminated or truncated):
        action = policy_fn(obs, env)
        obs, reward, terminated, truncated, info = env.step(action)
        
    metrics = env.compute_metrics()
    metrics['Strategy'] = strategy_name
    return metrics

# 1. Chiến lược Equal Weight (1/N cổ phiếu, 0% cash)
def equal_weight_policy(obs, env):
    action = np.zeros(env.action_dim, dtype=np.float32)
    action[1:] = 1.0 / env.num_stocks  # Phân bổ đều cho các cổ phiếu
    return action

# 2. Chiến lược 100% Cash
def cash_only_policy(obs, env):
    action = np.zeros(env.action_dim, dtype=np.float32)
    action[0] = 1.0  # 100% Tiền mặt
    return action

# 3. Chiến lược Tái cân bằng Ngẫu nhiên (Dirichlet distribution)
def random_policy(obs, env):
    return np.random.dirichlet(np.ones(env.action_dim)).astype(np.float32)

# Thực thi đánh giá
results = []

# Run Equal Weight
m_ew = evaluate_strategy(test_env, 'Equal Weight (1/N)', equal_weight_policy)
history_ew = test_env.asset_memory.copy()
results.append(m_ew)

# Run Cash Only
m_cash = evaluate_strategy(test_env, 'Cash Only (100%)', cash_only_policy)
history_cash = test_env.asset_memory.copy()
results.append(m_cash)

# Run Random
m_rnd = evaluate_strategy(test_env, 'Random Rebalancing', random_policy)
history_rnd = test_env.asset_memory.copy()
results.append(m_rnd)

# Hiển thị bảng tổng hợp so sánh
df_results = pd.DataFrame(results)
df_results.set_index('Strategy', inplace=True)
print('📊 BẢNG KẾT QUẢ ĐỐI CHIẾU CÁC CHIẾN LƯỢC TRÊN TẬP TEST:')
print(df_results.to_string())


Initial portfolio value:100000.0
Final portfolio value: 113949.796875
Final accumulative portfolio value: 1.13949796875
Maximum DrawDown: -0.14846865776122942
Sharpe ratio: 1.6438925983041726
Initial portfolio value:100000.0
Final portfolio value: 100000.0
Final accumulative portfolio value: 1.0
Maximum DrawDown: 0.0
Sharpe ratio: 0.0
Initial portfolio value:100000.0
Final portfolio value: 68297.9921875
Final accumulative portfolio value: 0.682979921875
Maximum DrawDown: -0.31800864590499056
Sharpe ratio: -3.480405416634423
📊 BẢNG KẾT QUẢ ĐỐI CHIẾU CÁC CHIẾN LƯỢC TRÊN TẬP TEST:
                    Initial Value ($)  Final Value ($)  Cumulative Return (%)  Annualized Return (%)  Annualized Volatility (%)  Sharpe Ratio  Max Drawdown (%)
Strategy                                                                                                                                                       
Equal Weight (1/N)           100000.0    113949.796875              13.949797              13.3

In [ ]:
# ====================================================================
# HÀM TRỰC QUAN HÓA TOÀN DIỆN KẾT QUẢ ĐẦU TƯ CÁC MÔ HÌNH (PORTFOLIO PLOTTING)
# ====================================================================

def plot_investment_results(
    histories: Optional[Dict[str, Union[List[float], np.ndarray]]] = None,
    df_metrics: Optional[pd.DataFrame] = None,
    initial_amount: float = 100_000.0,
    title: str = "ĐỐI CHIẾU HIỆU SUẤT TĂNG TRƯỞNG & RỦI RO CÁC MÔ HÌNH ĐẦU TƯ",
    save_path: Optional[str] = "investment_results_comparison.png",
    show_plot: bool = True
) -> Tuple[plt.Figure, pd.DataFrame]:
    """
    Hàm trực quan hóa chuyên sâu kết quả đầu tư của các mô hình và chiến lược benchmark:
    - Panel 1: Đường cong tăng trưởng giá trị tài sản danh mục (Portfolio Equity Curves)
    - Panel 2: Biểu đồ sụt giảm vốn tối đa theo thời gian (Underwater Drawdown Chart %)
    - Panel 3: Biểu đồ cột đối chiếu Lợi nhuận tích lũy (Cumulative Return %)
    - Panel 4: Biểu đồ cột đối chiếu Hệ số Sharpe (Sharpe Ratio)
    
    Trả về: (fig, summary_df) chứa đối tượng Figure và bảng DataFrame tổng hợp các chỉ số.
    """
    # Tự động thu thập các lịch sử tài sản nếu không truyền tham số
    if histories is None:
        histories = {}
        candidate_vars = [
            ('Equal Weight (1/N)', 'history_ew'),
            ('Cash Only (100%)', 'history_cash'),
            ('Random Rebalancing', 'history_rnd'),
            ('A2C', 'history_a2c'),
            ('PPO', 'history_ppo'),
            ('DDPG', 'history_ddpg'),
            ('TD3', 'history_td3'),
            ('SAC', 'history_sac'),
            ('TQC', 'history_tqc'),
            ('CrossQ', 'history_crossq'),
        ]
        caller_globals = globals()
        for name, var_name in candidate_vars:
            if var_name in caller_globals and caller_globals[var_name] is not None:
                if len(caller_globals[var_name]) > 0:
                    histories[name] = caller_globals[var_name]

    if not histories:
        raise ValueError("Không tìm thấy lịch sử tài sản để vẽ biểu đồ. Hãy đảm bảo ít nhất một mô hình đã được đánh giá.")

    # Bảng màu chuẩn phân tích định lượng cho từng chiến lược
    style_map = {
        'Equal Weight (1/N)': {'color': '#2ca02c', 'lw': 2.4, 'ls': '-'},
        'Benchmark: Equal Weight (1/N)': {'color': '#2ca02c', 'lw': 2.4, 'ls': '-'},
        'Cash Only (100%)': {'color': '#7f7f7f', 'lw': 1.8, 'ls': '--'},
        'Benchmark: Cash Only (100%)': {'color': '#7f7f7f', 'lw': 1.8, 'ls': '--'},
        'Random Rebalancing': {'color': '#d62728', 'lw': 1.5, 'ls': ':'},
        'PPO': {'color': '#1f77b4', 'lw': 2.2, 'ls': '-'},
        'PPO (Clipped Surrogate)': {'color': '#1f77b4', 'lw': 2.2, 'ls': '-'},
        'PPO Agent (SB3)': {'color': '#1f77b4', 'lw': 2.2, 'ls': '-'},
        'A2C': {'color': '#9467bd', 'lw': 1.8, 'ls': '-'},
        'A2C (On-Policy)': {'color': '#9467bd', 'lw': 1.8, 'ls': '-'},
        'DDPG': {'color': '#8c564b', 'lw': 1.8, 'ls': '-'},
        'DDPG (Deterministic)': {'color': '#8c564b', 'lw': 1.8, 'ls': '-'},
        'TD3': {'color': '#e377c2', 'lw': 2.0, 'ls': '-'},
        'TD3 (Twin Delayed)': {'color': '#e377c2', 'lw': 2.0, 'ls': '-'},
        'SAC': {'color': '#ff7f0e', 'lw': 2.2, 'ls': '-'},
        'SAC (Maximum Entropy)': {'color': '#ff7f0e', 'lw': 2.2, 'ls': '-'},
        'TQC': {'color': '#bcbd22', 'lw': 2.0, 'ls': '-'},
        'TQC (Truncated Quantiles)': {'color': '#bcbd22', 'lw': 2.0, 'ls': '-'},
        'CrossQ': {'color': '#17becf', 'lw': 2.0, 'ls': '-'},
        'CrossQ (Batch Norm DRL)': {'color': '#17becf', 'lw': 2.0, 'ls': '-'},
    }

    # Tính toán các chỉ số và Drawdown
    calculated_metrics = []
    drawdowns_dict = {}

    for name, history in histories.items():
        vals = np.array(history, dtype=np.float64)
        if len(vals) < 2:
            continue
        cummax = np.maximum.accumulate(vals)
        dd = (vals - cummax) / (cummax + 1e-8) * 100.0
        drawdowns_dict[name] = dd
        
        ret_series = np.diff(vals) / (vals[:-1] + 1e-8)
        tot_ret = (vals[-1] - vals[0]) / vals[0] * 100.0
        ann_ret = (((vals[-1] / vals[0]) ** (252.0 / len(vals))) - 1.0) * 100.0 if vals[-1] > 0 else -100.0
        ann_vol = np.std(ret_series) * np.sqrt(252.0) * 100.0 if len(ret_series) > 1 else 0.0
        sharpe = (ann_ret - 2.0) / (ann_vol + 1e-8) if ann_vol > 0 else 0.0
        max_dd = np.min(dd)

        calculated_metrics.append({
            'Strategy': name,
            'Initial Value ($)': vals[0],
            'Final Value ($)': vals[-1],
            'Cumulative Return (%)': tot_ret,
            'Annualized Return (%)': ann_ret,
            'Annualized Volatility (%)': ann_vol,
            'Sharpe Ratio': sharpe,
            'Max Drawdown (%)': abs(max_dd)
        })

    summary_df = pd.DataFrame(calculated_metrics)
    if not summary_df.empty:
        summary_df.set_index('Strategy', inplace=True)
        summary_df.sort_values(by='Sharpe Ratio', ascending=False, inplace=True)

    # Khởi tạo bố cục đồ thị 3 tầng trực quan
    fig = plt.figure(figsize=(15, 11))
    gs = fig.add_gridspec(3, 2, height_ratios=[1.6, 1.0, 1.1], hspace=0.38, wspace=0.25)
    fig.suptitle(title, fontsize=15, fontweight='bold', y=0.98)

    # Panel 1: Đường cong giá trị tài sản
    ax1 = fig.add_subplot(gs[0, :])
    for name, history in histories.items():
        st = style_map.get(name, {'color': None, 'lw': 1.8, 'ls': '-'})
        ax1.plot(history, label=name, color=st.get('color'), lw=st.get('lw', 1.8), linestyle=st.get('ls', '-'))
    ax1.axhline(initial_amount, color='black', linestyle=':', alpha=0.7, label=f'Vốn ban đầu (${initial_amount:,.0f})')
    ax1.set_title('1. ĐƯỜNG CONG TĂNG TRƯỞNG TÀI SẢN (PORTFOLIO EQUITY CURVES)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Ngày giao dịch (Trading Days)', fontsize=10)
    ax1.set_ylabel('Giá trị Danh mục ($)', fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=9)

    # Panel 2: Sụt giảm vốn Underwater Drawdown
    ax2 = fig.add_subplot(gs[1, :], sharex=ax1)
    for name, dd in drawdowns_dict.items():
        st = style_map.get(name, {'color': None, 'lw': 1.5, 'ls': '-'})
        ax2.plot(dd, label=name, color=st.get('color'), lw=st.get('lw', 1.5), linestyle=st.get('ls', '-'), alpha=0.85)
    ax2.axhline(0, color='black', linestyle='-', lw=0.8)
    ax2.set_title('2. MỨC ĐỘ SỤT GIẢM TỐI ĐA THEO THỜI GIAN (UNDERWATER DRAWDOWN %)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Ngày giao dịch (Trading Days)', fontsize=10)
    ax2.set_ylabel('Drawdown (%)', fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(top=1.0)

    # Panel 3: Lợi nhuận tích lũy
    ax3 = fig.add_subplot(gs[2, 0])
    if not summary_df.empty:
        rets = summary_df['Cumulative Return (%)']
        colors = ['#2ca02c' if x >= 0 else '#d62728' for x in rets]
        bars = ax3.barh(rets.index, rets.values, color=colors, alpha=0.85, edgecolor='black', lw=0.6)
        ax3.axvline(0, color='black', linestyle='--', lw=0.8)
        ax3.set_title('3. LỢI NHUẬN TÍCH LŨY (CUMULATIVE RETURN %)', fontsize=11, fontweight='bold')
        ax3.set_xlabel('Lợi nhuận (%)', fontsize=10)
        ax3.grid(True, alpha=0.3, axis='x')
        for bar in bars:
            val = bar.get_width()
            align = 'left' if val >= 0 else 'right'
            offset = max(abs(val) * 0.05, 0.3) * (1 if val >= 0 else -1)
            ax3.text(val + offset, bar.get_y() + bar.get_height()/2, f'{val:+.1f}%',
                     va='center', ha=align, fontsize=8.5, fontweight='bold')

    # Panel 4: Sharpe Ratio
    ax4 = fig.add_subplot(gs[2, 1])
    if not summary_df.empty:
        sharpes = summary_df['Sharpe Ratio']
        colors = ['#1f77b4' if x >= 1.0 else ('#ff7f0e' if x >= 0 else '#d62728') for x in sharpes]
        bars = ax4.barh(sharpes.index, sharpes.values, color=colors, alpha=0.85, edgecolor='black', lw=0.6)
        ax4.axvline(0, color='black', linestyle='--', lw=0.8)
        ax4.axvline(1.0, color='#2ca02c', linestyle=':', lw=1.2, label='Sharpe = 1.0')
        ax4.set_title('4. HỆ SỐ SHARPE (SHARPE RATIO)', fontsize=11, fontweight='bold')
        ax4.set_xlabel('Sharpe Ratio', fontsize=10)
        ax4.grid(True, alpha=0.3, axis='x')
        ax4.legend(loc='lower right', fontsize=8.5)
        for bar in bars:
            val = bar.get_width()
            align = 'left' if val >= 0 else 'right'
            offset = max(abs(val) * 0.05, 0.04) * (1 if val >= 0 else -1)
            ax4.text(val + offset, bar.get_y() + bar.get_height()/2, f'{val:.2f}',
                     va='center', ha=align, fontsize=8.5, fontweight='bold')

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"💾 Đã lưu biểu đồ kết quả đầu tư vào: {save_path}")

    if show_plot:
        plt.show()

    return fig, summary_df

# Áp dụng hàm vẽ kết quả cho 3 chiến lược Benchmark
benchmark_histories = {
    'Equal Weight (1/N)': history_ew,
    'Cash Only (100%)': history_cash,
    'Random Rebalancing': history_rnd
}

# fig_bench, df_bench = plot_investment_results(
#     histories=benchmark_histories,
#     df_metrics=df_results,
#     title='SO SÁNH HIỆU SUẤT CÁC CHIẾN LƯỢC BENCHMARK TRÊN TẬP TEST',
#     save_path='benchmark_results_comparison.png'
# )


💾 Đã lưu biểu đồ kết quả đầu tư vào: benchmark_results_comparison.png


---
## 7. Tích hợp Với Stable-Baselines3 (PPO / SAC / A2C)

### Vấn đề tương thích kiến trúc:
- `FinRLPortfolioOptimizationEnv` xuất ra không gian quan sát dạng Tensor 3D $(F, N, M)$ và yêu cầu action thuộc mặt phẳng Simplex $\sum w_i = 1, w_i \ge 0$.
- Mạng `MlpPolicy` chuẩn trong **Stable-Baselines3** yêu cầu không gian quan sát 1 chiều phẳng (Flattened 1D vector).
- Do đó, ta tạo một lớp **Gymnasium Wrapper: `SB3PortfolioEnvWrapper`**:
  1. Trải phẳng (Flatten) quan sát $(F \times N \times M)$ thành vector $1D$ kích thước $(F \times N \times M)$.
  2. Tự động chuyển đổi đầu ra Gaussian Policy của SB3 thành vector xác suất Simplex bằng Softmax.
  3. Cung cấp API chuẩn cho `DummyVecEnv`.


In [8]:
class SB3PortfolioEnvWrapper(gym.ObservationWrapper):
    """
    Wrapper biến đổi môi trường FinRLPortfolioOptimizationEnv tương thích 100% với Stable-Baselines3.
    - Làm phẳng observation space thành Box(F * N * M,)
    - Action space giữ nguyên Box(N + 1,)
    """
    def __init__(self, env: FinRLPortfolioOptimizationEnv):
        super().__init__(env)
        flat_dim = env.num_features * env.num_stocks * env.time_window
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(flat_dim,), dtype=np.float32
        )
        
    def observation(self, observation: np.ndarray) -> np.ndarray:
        if isinstance(observation, dict):
            obs = observation['state']
        else:
            obs = observation
        return obs.flatten().astype(np.float32)

print('✅ Lớp SB3PortfolioEnvWrapper đã sẵn sàng tích hợp!')


✅ Lớp SB3PortfolioEnvWrapper đã sẵn sàng tích hợp!


In [9]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

# Đóng gói môi trường bằng Wrapper và VecEnv
wrapped_train_env = DummyVecEnv([lambda: SB3PortfolioEnvWrapper(train_env)])
wrapped_test_env = DummyVecEnv([lambda: SB3PortfolioEnvWrapper(test_env)])

print('>>> Khởi tạo mô hình PPO (Proximal Policy Optimization)...')
model = PPO(
    policy='MlpPolicy',
    env=wrapped_train_env,
    learning_rate=3e-4,
    n_steps=128,
    batch_size=32,
    gamma=0.99,
    ent_coef=0.01,
    verbose=0,
    seed=SEED
)

print('>>> Bắt đầu huấn luyện PPO trên 2,000 timesteps...')
model.learn(total_timesteps=2000)
print('✅ Hoàn thành huấn luyện PPO!')

# Đánh giá PPO trên tập Test
def ppo_policy(obs, env):
    flat_obs = obs.flatten().astype(np.float32)
    action, _states = model.predict(flat_obs, deterministic=True)
    return action

m_ppo = evaluate_strategy(test_env, 'PPO Agent (SB3)', ppo_policy)
print('\n📊 KẾT QUẢ ĐÁNH GIÁ PPO AGENT:')
for k, v in m_ppo.items():
    print(f'  + {k}: {v}')


>>> Khởi tạo mô hình PPO (Proximal Policy Optimization)...
>>> Bắt đầu huấn luyện PPO trên 2,000 timesteps...
Initial portfolio value:100000.0
Final portfolio value: 55731.2578125
Final accumulative portfolio value: 0.557312578125
Maximum DrawDown: -0.47055459885201545
Sharpe ratio: -2.932687777188244
Initial portfolio value:100000.0
Final portfolio value: 53928.28125
Final accumulative portfolio value: 0.5392828125
Maximum DrawDown: -0.47990973716033714
Sharpe ratio: -3.1694523235955643
Initial portfolio value:100000.0
Final portfolio value: 55062.0234375
Final accumulative portfolio value: 0.550620234375
Maximum DrawDown: -0.47146953134757075
Sharpe ratio: -3.0537555958224356
✅ Hoàn thành huấn luyện PPO!
Initial portfolio value:100000.0
Final portfolio value: 111130.9296875
Final accumulative portfolio value: 1.111309296875
Maximum DrawDown: -0.12050707295189489
Sharpe ratio: 1.7546740285905926

📊 KẾT QUẢ ĐÁNH GIÁ PPO AGENT:
  + Initial Value ($): 100000.0
  + Final Value ($): 111130

## Khởi tạo thử nghiệm toàn bộ mô hình áp dụng được cho bài toán

Làm rõ và tái hiện lại tổng bộ thiết kế kiến trúc các mô hình thuật toán chính của mỗi mô hình

### Mô hình A2C

In [10]:
# ====================================================================
# 1. MÔ HÌNH A2C (Advantage Actor-Critic)
# ====================================================================
from stable_baselines3 import A2C

print('='*70)
print('>>> [1/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN A2C (Advantage Actor-Critic)')
print('='*70)

# 1. Khởi tạo mô hình A2C theo tham số chuẩn FinRL (A2C_PARAMS: n_steps=5, ent_coef=0.01, lr=0.0007)
a2c_model = A2C(
    policy='MlpPolicy',
    env=wrapped_train_env,
    learning_rate=0.0007,
    n_steps=5,                 # Cập nhật on-policy sau mỗi 5 bước (chu kỳ ngắn)
    gamma=0.99,
    ent_coef=0.01,             # Hệ số Entropy bonus chống kẹt cực tiểu cục bộ
    vf_coef=0.5,               # Trọng số Value loss
    max_grad_norm=0.5,         # Gradient clipping
    verbose=0,
    seed=SEED
)

# 2. Bóc tách kiến trúc mạng nơ-ron bên trong
print('\n📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA A2C:')
print(f'  • Policy Class:       {a2c_model.policy.__class__.__name__}')
print(f'  • Feature Extractor:  {a2c_model.policy.features_extractor.__class__.__name__} (Input: {a2c_model.observation_space.shape[0]} -> Output: {a2c_model.policy.features_dim})')
print(f'  • Shared/MLP Extractor: {a2c_model.policy.mlp_extractor}')
print(f'  • Actor Head (action_net): {a2c_model.policy.action_net} (Xuất vector kỳ vọng μ cho {train_env.action_dim} vị thế)')
print(f'  • Critic Head (value_net): {a2c_model.policy.value_net} (Xuất giá trị trạng thái vô hướng V(s))')
print(f'  • Action Distribution: {a2c_model.policy.action_dist.__class__.__name__} (DiagGaussian với tham số log_std độc lập)')

# 3. Huấn luyện mô hình
print('\n🚀 Bắt đầu huấn luyện A2C trên 2,000 timesteps...')
a2c_model.learn(total_timesteps=2000)
print('✅ Huấn luyện A2C hoàn tất!')

# 4. Đánh giá trên tập Test Out-of-Sample
def a2c_policy(obs, env):
    flat_obs = obs.flatten().astype(np.float32)
    action, _ = a2c_model.predict(flat_obs, deterministic=True)
    return action

m_a2c = evaluate_strategy(test_env, 'A2C', a2c_policy)
history_a2c = test_env.asset_memory.copy()
sb3_all_results = [m_ew, m_cash, m_rnd, m_a2c]

print('\n📊 KẾT QUẢ ĐÁNH GIÁ THỰC TẾ A2C:')
for k, v in m_a2c.items():
    if k != 'Strategy':
        print(f'  • {k}: {v}')

print('\n💡 NHẬN XÉT ĐẶC THÙ CHO BÀI TOÁN PORTFOLIO MANAGEMENT:')
print('  - A2C là thuật toán on-policy đồng bộ cập nhật theo batch ngắn (n_steps=5).')
print('  - Trong bài toán PM, do chu kỳ n_steps ngắn, A2C phản ứng nhanh với biến động giá ngắn hạn nhưng dễ bị nhiễu cao.')
print('  - Phân phối Gaussian của Actor được chiếu qua Softmax Simplex ở môi trường để bảo toàn tổng tỷ trọng = 1.')


>>> [1/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN A2C (Advantage Actor-Critic)

📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA A2C:
  • Policy Class:       ActorCriticPolicy
  • Feature Extractor:  FlattenExtractor (Input: 900 -> Output: 900)
  • Shared/MLP Extractor: MlpExtractor(
  (policy_net): Sequential(
    (0): Linear(in_features=900, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
  )
  (value_net): Sequential(
    (0): Linear(in_features=900, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
  )
)
  • Actor Head (action_net): Linear(in_features=64, out_features=11, bias=True) (Xuất vector kỳ vọng μ cho 11 vị thế)
  • Critic Head (value_net): Linear(in_features=64, out_features=1, bias=True) (Xuất giá trị trạng thái vô hướng V(s))
  • Action Distribution: DiagGaussianDistribution (DiagGaussian với tham số log_std độc lập)

🚀 Bắt đầu huấn luyện A2C tr

### Mô hình PPO

In [11]:
# ====================================================================
# 2. MÔ HÌNH PPO (Proximal Policy Optimization)
# ====================================================================
from stable_baselines3 import PPO

print('='*70)
print('>>> [2/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN PPO (Proximal Policy Optimization)')
print('='*70)

# 1. Khởi tạo mô hình PPO theo chuẩn FinRL (PPO_PARAMS: n_steps=128/2048, batch_size=32/64, lr=0.00025)
ppo_model = PPO(
    policy='MlpPolicy',
    env=wrapped_train_env,
    learning_rate=0.00025,
    n_steps=128,               # Kích thước rollout buffer
    batch_size=32,             # Mini-batch size tối ưu hóa
    n_epochs=10,               # Số vòng lặp cập nhật trên mỗi rollout buffer
    gamma=0.99,
    gae_lambda=0.95,           # Hệ số Generalized Advantage Estimation
    clip_range=0.2,            # Ngưỡng xén tỷ số xác suất: 1 ± 0.2
    ent_coef=0.01,             # Entropy bonus thúc đẩy đa dạng hóa danh mục
    vf_coef=0.5,
    verbose=0,
    seed=SEED
)

# 2. Bóc tách kiến trúc mạng nơ-ron bên trong
print('\n📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA PPO:')
print(f'  • Policy Class:       {ppo_model.policy.__class__.__name__}')
print(f'  • Feature Extractor:  {ppo_model.policy.features_extractor.__class__.__name__}')
print(f'  • MLP Extractor (Tách nhánh Policy & Value riêng biệt):')
print(f'    - Policy Net (pi):  {ppo_model.policy.mlp_extractor.policy_net}')
print(f'    - Value Net (vf):   {ppo_model.policy.mlp_extractor.value_net}')
print(f'  • Action Head:        {ppo_model.policy.action_net} -> Output dimension: {train_env.action_dim}')
print(f'  • Value Head:         {ppo_model.policy.value_net} -> Output dimension: 1')
print(f'  • Cơ chế Clipped Surrogate: Ngăn chặn cập nhật chính sách quá lớn: r_t(θ) ∈ [0.8, 1.2]')

# 3. Huấn luyện mô hình
print('\n🚀 Bắt đầu huấn luyện PPO trên 2,000 timesteps...')
ppo_model.learn(total_timesteps=2000)
print('✅ Huấn luyện PPO hoàn tất!')

# 4. Đánh giá trên tập Test Out-of-Sample
def ppo_policy_fn(obs, env):
    flat_obs = obs.flatten().astype(np.float32)
    action, _ = ppo_model.predict(flat_obs, deterministic=True)
    return action

m_ppo = evaluate_strategy(test_env, 'PPO', ppo_policy_fn)
history_ppo = test_env.asset_memory.copy()
sb3_all_results.append(m_ppo)

print('\n📊 KẾT QUẢ ĐÁNH GIÁ THỰC TẾ PPO:')
for k, v in m_ppo.items():
    if k != 'Strategy':
        print(f'  • {k}: {v}')

print('\n💡 NHẬN XÉT ĐẶC THÙ CHO BÀI TOÁN PORTFOLIO MANAGEMENT:')
print('  - PPO ổn định hơn hẳn A2C nhờ cơ chế Surrogate Clipping, hạn chế việc thay đổi đột ngột tỷ trọng danh mục.')
print('  - Entropy bonus giúp tránh hiện tượng phân bổ 100% vào 1 mã duy nhất khi mới bắt đầu học.')


>>> [2/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN PPO (Proximal Policy Optimization)

📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA PPO:
  • Policy Class:       ActorCriticPolicy
  • Feature Extractor:  FlattenExtractor
  • MLP Extractor (Tách nhánh Policy & Value riêng biệt):
    - Policy Net (pi):  Sequential(
  (0): Linear(in_features=900, out_features=64, bias=True)
  (1): Tanh()
  (2): Linear(in_features=64, out_features=64, bias=True)
  (3): Tanh()
)
    - Value Net (vf):   Sequential(
  (0): Linear(in_features=900, out_features=64, bias=True)
  (1): Tanh()
  (2): Linear(in_features=64, out_features=64, bias=True)
  (3): Tanh()
)
  • Action Head:        Linear(in_features=64, out_features=11, bias=True) -> Output dimension: 11
  • Value Head:         Linear(in_features=64, out_features=1, bias=True) -> Output dimension: 1
  • Cơ chế Clipped Surrogate: Ngăn chặn cập nhật chính sách quá lớn: r_t(θ) ∈ [0.8, 1.2]

🚀 Bắt đầu huấn luyện PPO trên 2,000 timesteps...
Initial portfolio value:100000.0

### Mô hình DDPG

In [12]:
# ====================================================================
# 3. MÔ HÌNH DDPG (Deep Deterministic Policy Gradient)
# ====================================================================
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import OrnsteinUhlenbeckActionNoise

print('='*70)
print('>>> [3/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN DDPG (Deep Deterministic Policy Gradient)')
print('='*70)

# 1. Thiết lập Nhiễu hành động (Ornstein-Uhlenbeck Action Noise) chuẩn FinRL
n_actions = train_env.action_dim
ou_noise = OrnsteinUhlenbeckActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

# 2. Khởi tạo mô hình DDPG (buffer_size an toàn 10,000 tránh tràn RAM)
ddpg_model = DDPG(
    policy='MlpPolicy',
    env=wrapped_train_env,
    learning_rate=0.001,
    buffer_size=10_000,        # Replay Buffer lưu trữ chuyển trạng thái (s, a, r, s')
    batch_size=64,
    tau=0.005,                 # Hệ số Polyak update mạng target
    gamma=0.99,
    action_noise=ou_noise,     # Nhiễu khám phá tự tương quan theo thời gian
    verbose=0,
    seed=SEED
)

# 3. Bóc tách kiến trúc mạng nơ-ron bên trong
print('\n📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA DDPG:')
print(f'  • Policy Class:          {ddpg_model.policy.__class__.__name__}')
print(f'  • Actor (Deterministic): {ddpg_model.policy.actor}')
print(f'    - Lớp cuối Actor:      Linear -> Tanh() (Ép hành động về [-1, 1], sau đó co giãn về [0, 1])')
print(f'  • Critic (Q-network):    {ddpg_model.policy.critic}')
print(f'    - Đầu vào Critic:      Ghép nối [State, Action] kích thước {a2c_model.observation_space.shape[0]} + {n_actions}')
print(f'  • Target Networks:       actor_target & critic_target cập nhật chậm qua Polyak (tau=0.005)')

# 4. Huấn luyện mô hình
print('\n🚀 Bắt đầu huấn luyện DDPG trên 2,000 timesteps...')
ddpg_model.learn(total_timesteps=2000)
print('✅ Huấn luyện DDPG hoàn tất!')

# 5. Đánh giá trên tập Test Out-of-Sample
def ddpg_policy_fn(obs, env):
    flat_obs = obs.flatten().astype(np.float32)
    action, _ = ddpg_model.predict(flat_obs, deterministic=True)
    return action

m_ddpg = evaluate_strategy(test_env, 'DDPG', ddpg_policy_fn)
history_ddpg = test_env.asset_memory.copy()
sb3_all_results.append(m_ddpg)

print('\n📊 KẾT QUẢ ĐÁNH GIÁ THỰC TẾ DDPG:')
for k, v in m_ddpg.items():
    if k != 'Strategy':
        print(f'  • {k}: {v}')

print('\n💡 NHẬN XÉT ĐẶC THÙ CHO BÀI TOÁN PORTFOLIO MANAGEMENT:')
print('  - DDPG xuất ra hành động tất định, loại bỏ hoàn toàn độ ngẫu nhiên khi đưa ra quyết định.')
print('  - Tuy nhiên, DDPG chỉ có 1 Critic duy nhất nên rất dễ bị hiện tượng Overestimation Bias (ước lượng lợi nhuận quá mức).')


>>> [3/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN DDPG (Deep Deterministic Policy Gradient)

📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA DDPG:
  • Policy Class:          TD3Policy
  • Actor (Deterministic): Actor(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mu): Sequential(
    (0): Linear(in_features=900, out_features=400, bias=True)
    (1): ReLU()
    (2): Linear(in_features=400, out_features=300, bias=True)
    (3): ReLU()
    (4): Linear(in_features=300, out_features=11, bias=True)
    (5): Tanh()
  )
)
    - Lớp cuối Actor:      Linear -> Tanh() (Ép hành động về [-1, 1], sau đó co giãn về [0, 1])
  • Critic (Q-network):    ContinuousCritic(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (qf0): Sequential(
    (0): Linear(in_features=911, out_features=400, bias=True)
    (1): ReLU()
    (2): Linear(in_features=400, out_features=300, bias=True)
    (3): ReLU()
    (4): Linear(in_features=30

### Mô hình TD3

In [13]:
# ====================================================================
# 4. MÔ HÌNH TD3 (Twin Delayed DDPG)
# ====================================================================
from stable_baselines3 import TD3
from stable_baselines3.common.noise import NormalActionNoise

print('='*70)
print('>>> [4/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN TD3 (Twin Delayed DDPG)')
print('='*70)

# 1. Nhiễu hành động Gaussian cho TD3
gauss_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

# 2. Khởi tạo mô hình TD3
td3_model = TD3(
    policy='MlpPolicy',
    env=wrapped_train_env,
    learning_rate=0.001,
    buffer_size=10_000,        # Safe buffer size
    batch_size=64,
    tau=0.005,
    gamma=0.99,
    policy_delay=2,            # Cập nhật Actor 1 lần sau mỗi 2 lần cập nhật Critic
    target_policy_noise=0.2,   # Nhiễu làm mịn mục tiêu (Target Policy Smoothing)
    target_noise_clip=0.5,
    action_noise=gauss_noise,
    verbose=0,
    seed=SEED
)

# 3. Bóc tách kiến trúc mạng nơ-ron bên trong
print('\n📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA TD3:')
print(f'  • Policy Class:       {td3_model.policy.__class__.__name__}')
print(f'  • Actor Network:      {td3_model.policy.actor}')
print(f'  • Twin Critics (2 mạng Q độc lập Q1 và Q2):') 
print(f'    - Critic 1 (qf0):   {td3_model.policy.critic.q_networks[0]}')
print(f'    - Critic 2 (qf1):   {td3_model.policy.critic.q_networks[1]}')
print('  • Cơ chế Clipped Double Q: Lấy giá trị nhỏ nhất min(Q1, Q2) khi tính Bellman target -> Trị tận gốc overestimation!')

# 4. Huấn luyện mô hình
print('\n🚀 Bắt đầu huấn luyện TD3 trên 2,000 timesteps...')
td3_model.learn(total_timesteps=2000)
print('✅ Huấn luyện TD3 hoàn tất!')

# 5. Đánh giá trên tập Test Out-of-Sample
def td3_policy_fn(obs, env):
    flat_obs = obs.flatten().astype(np.float32)
    action, _ = td3_model.predict(flat_obs, deterministic=True)
    return action

m_td3 = evaluate_strategy(test_env, 'TD3', td3_policy_fn)
history_td3 = test_env.asset_memory.copy()
sb3_all_results.append(m_td3)

print('\n📊 KẾT QUẢ ĐÁNH GIÁ THỰC TẾ TD3:')
for k, v in m_td3.items():
    if k != 'Strategy':
        print(f'  • {k}: {v}')

print('\n💡 NHẬN XÉT ĐẶC THÙ CHO BÀI TOÁN PORTFOLIO MANAGEMENT:')
print('  - TD3 cải thiện vượt trội so với DDPG trong môi trường tài chính vì loại bỏ sự ảo tưởng về lợi nhuận qua Twin Critics.')
print('  - Target Policy Smoothing giúp danh mục chống lại các điểm cực đoan của hàm Q đối với các mã cổ phiếu rủi ro.')


>>> [4/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN TD3 (Twin Delayed DDPG)

📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA TD3:
  • Policy Class:       TD3Policy
  • Actor Network:      Actor(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mu): Sequential(
    (0): Linear(in_features=900, out_features=400, bias=True)
    (1): ReLU()
    (2): Linear(in_features=400, out_features=300, bias=True)
    (3): ReLU()
    (4): Linear(in_features=300, out_features=11, bias=True)
    (5): Tanh()
  )
)
  • Twin Critics (2 mạng Q độc lập Q1 và Q2):
    - Critic 1 (qf0):   Sequential(
  (0): Linear(in_features=911, out_features=400, bias=True)
  (1): ReLU()
  (2): Linear(in_features=400, out_features=300, bias=True)
  (3): ReLU()
  (4): Linear(in_features=300, out_features=1, bias=True)
)
    - Critic 2 (qf1):   Sequential(
  (0): Linear(in_features=911, out_features=400, bias=True)
  (1): ReLU()
  (2): Linear(in_features=400, out_features=300, bias=True)
  (3): 

### Mô hình SAC

In [14]:
# ====================================================================
# 5. MÔ HÌNH SAC (Soft Actor-Critic)
# ====================================================================
from stable_baselines3 import SAC

print('='*70)
print('>>> [5/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN SAC (Soft Actor-Critic)')
print('='*70)

# 1. Khởi tạo mô hình SAC với Auto Entropy Tuning
sac_model = SAC(
    policy='MlpPolicy',
    env=wrapped_train_env,
    learning_rate=0.0003,
    buffer_size=10_000,
    batch_size=64,
    tau=0.005,
    gamma=0.99,
    ent_coef='auto',           # Tự động tối ưu hóa nhiệt độ Entropy alpha theo target entropy
    learning_starts=100,
    verbose=0,
    seed=SEED
)

# 2. Bóc tách kiến trúc mạng nơ-ron bên trong
print('\n📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA SAC:')
print(f'  • Policy Class:       {sac_model.policy.__class__.__name__}')
print(f'  • Stochastic Actor:   {sac_model.policy.actor}')
print(f'    - Nhánh mu:         {sac_model.policy.actor.mu}')
print(f'    - Nhánh log_std:    {sac_model.policy.actor.log_std}')
print(f'  • Action Distribution: SquashedDiagGaussianDistribution (Áp dụng Tanh(u) + Jacobian log-prob correction)')
print(f'  • Twin Soft Critics:  2 mạng Critic Q1 và Q2 có tính thành phần -alpha * log pi(a|s)')
print(f'  • Tham số Entropy alpha học được: log_ent_coef = {sac_model.log_ent_coef.item():.4f}')

# 3. Huấn luyện mô hình
print('\n🚀 Bắt đầu huấn luyện SAC trên 2,000 timesteps...')
sac_model.learn(total_timesteps=2000)
print('✅ Huấn luyện SAC hoàn tất!')

# 4. Đánh giá trên tập Test Out-of-Sample
def sac_policy_fn(obs, env):
    flat_obs = obs.flatten().astype(np.float32)
    action, _ = sac_model.predict(flat_obs, deterministic=True)
    return action

m_sac = evaluate_strategy(test_env, 'SAC', sac_policy_fn)
history_sac = test_env.asset_memory.copy()
sb3_all_results.append(m_sac)

print('\n📊 KẾT QUẢ ĐÁNH GIÁ THỰC TẾ SAC:')
for k, v in m_sac.items():
    if k != 'Strategy':
        print(f'  • {k}: {v}')

print('\n💡 NHẬN XÉT ĐẶC THÙ CHO BÀI TOÁN PORTFOLIO MANAGEMENT:')
print('  - Nguyên lý Maximum Entropy của SAC là công cụ tự nhiên tuyệt vời cho đa dạng hóa danh mục (Diversification).')
print('  - Thay vì chỉ tìm một danh mục tối ưu đơn lẻ, SAC tìm một chính sách phân bổ có entropy cao, giảm thiểu rủi ro tập trung.')


>>> [5/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN SAC (Soft Actor-Critic)

📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA SAC:
  • Policy Class:       SACPolicy
  • Stochastic Actor:   Actor(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (latent_pi): Sequential(
    (0): Linear(in_features=900, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
  )
  (mu): Linear(in_features=256, out_features=11, bias=True)
  (log_std): Linear(in_features=256, out_features=11, bias=True)
)
    - Nhánh mu:         Linear(in_features=256, out_features=11, bias=True)
    - Nhánh log_std:    Linear(in_features=256, out_features=11, bias=True)
  • Action Distribution: SquashedDiagGaussianDistribution (Áp dụng Tanh(u) + Jacobian log-prob correction)
  • Twin Soft Critics:  2 mạng Critic Q1 và Q2 có tính thành phần -alpha * log pi(a|s)
  • Tham số Entropy alpha học được: log_ent_coef = 0.0000

🚀 Bắt đ

### Mô hình TQC

In [15]:
# ====================================================================
# 6. MÔ HÌNH TQC (Truncated Quantile Critics)
# ====================================================================
from sb3_contrib import TQC

print('='*70)
print('>>> [6/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN TQC (Truncated Quantile Critics)')
print('='*70)

# 1. Khởi tạo mô hình TQC (sb3-contrib)
tqc_model = TQC(
    policy='MlpPolicy',
    env=wrapped_train_env,
    learning_rate=0.0003,
    buffer_size=10_000,
    batch_size=64,
    gamma=0.99,
    top_quantiles_to_drop_per_net=2, # Cắt bỏ 2 quantiles cao nhất của mỗi mạng Critic
    verbose=0,
    seed=SEED
)

# 2. Bóc tách kiến trúc mạng nơ-ron bên trong
print('\n📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA TQC:')
print(f'  • Policy Class:          {tqc_model.policy.__class__.__name__}')
print(f'  • Quantile Critic:       {tqc_model.policy.critic.__class__.__name__}')
# print(f'    - Số lượng Critic Nets: {tqc_model.n_critics}')
# print(f'    - Số Quantiles / Net:   {tqc_model.quantiles_total // tqc_model.n_critics}')
# print(f'    - Tổng số Quantiles:    {tqc_model.quantiles_total}')
# print(f'    - Số Quantiles bị cắt:  {tqc_model.top_quantiles_to_drop_per_net * tqc_model.n_critics} (loại trừ các giá trị kỳ vọng ảo)')
print('  • Cơ chế Phân vị: Ước lượng toàn bộ phân phối xác suất của lợi nhuận thay vì chỉ ước lượng giá trị trung bình!')

# 3. Huấn luyện mô hình
print('\n🚀 Bắt đầu huấn luyện TQC trên 2,000 timesteps...')
tqc_model.learn(total_timesteps=2000)
print('✅ Huấn luyện TQC hoàn tất!')

# 4. Đánh giá trên tập Test Out-of-Sample
def tqc_policy_fn(obs, env):
    flat_obs = obs.flatten().astype(np.float32)
    action, _ = tqc_model.predict(flat_obs, deterministic=True)
    return action

m_tqc = evaluate_strategy(test_env, 'TQC', tqc_policy_fn)
history_tqc = test_env.asset_memory.copy()
sb3_all_results.append(m_tqc)

print('\n📊 KẾT QUẢ ĐÁNH GIÁ THỰC TẾ TQC:')
for k, v in m_tqc.items():
    if k != 'Strategy':
        print(f'  • {k}: {v}')

print('\n💡 NHẬN XÉT ĐẶC THÙ CHO BÀI TOÁN PORTFOLIO MANAGEMENT:')
print('  - TQC là thuật toán đoạt giải tiên tiến nhất cho thị trường tài chính biến động (đặc biệt Crypto / Volatile stocks).')
print('  - Việc cắt bỏ quantiles cao giúp loại trừ thiên kiến lạc quan tếu (Over-optimism) trước các đợt tăng giá ảo ngắn hạn.')


>>> [6/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN TQC (Truncated Quantile Critics)

📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA TQC:
  • Policy Class:          TQCPolicy
  • Quantile Critic:       Critic
  • Cơ chế Phân vị: Ước lượng toàn bộ phân phối xác suất của lợi nhuận thay vì chỉ ước lượng giá trị trung bình!

🚀 Bắt đầu huấn luyện TQC trên 2,000 timesteps...
Initial portfolio value:100000.0
Final portfolio value: 63952.4375
Final accumulative portfolio value: 0.639524375
Maximum DrawDown: -0.38636331707463667
Sharpe ratio: -2.1620952554532775
Initial portfolio value:100000.0
Final portfolio value: 68605.5
Final accumulative portfolio value: 0.686055
Maximum DrawDown: -0.33997012905736457
Sharpe ratio: -1.9264025583815774
Initial portfolio value:100000.0
Final portfolio value: 68756.0
Final accumulative portfolio value: 0.68756
Maximum DrawDown: -0.3427193597221313
Sharpe ratio: -1.94170119752096
✅ Huấn luyện TQC hoàn tất!
Initial portfolio value:100000.0
Final portfolio value: 112548.7656

### Mô hình CrossQ

In [16]:
# ====================================================================
# 7. MÔ HÌNH CrossQ (Batch Normalization in Deep RL)
# ====================================================================
from sb3_contrib import CrossQ

print('='*70)
print('>>> [7/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN CrossQ (ICLR 2024)')
print('='*70)

# 1. Khởi tạo mô hình CrossQ (sb3-contrib)
crossq_model = CrossQ(
    policy='MlpPolicy',
    env=wrapped_train_env,
    learning_rate=0.001,
    buffer_size=10_000,
    batch_size=64,
    gamma=0.99,
    verbose=0,
    seed=SEED
)

# 2. Bóc tách kiến trúc mạng nơ-ron bên trong
print('\n📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA CrossQ:')
print(f'  • Policy Class:       {crossq_model.policy.__class__.__name__}')
print(f'  • Cơ chế Đột phá:     HOÀN TOÀN KHÔNG SỬ DỤNG TARGET NETWORKS (No Target Delay)!') 
print(f'  • Batch Normalization: Áp dụng trực tiếp vào Critic: {crossq_model.policy.critic}')
print('  • Cross-Batching: Gộp chung (s_t, a_t) và (s_{t+1}, a_{t+1}) trong cùng một lượt forward để chuẩn hóa thống kê.')

# 3. Huấn luyện mô hình
print('\n🚀 Bắt đầu huấn luyện CrossQ trên 2,000 timesteps...')
crossq_model.learn(total_timesteps=2000)
print('✅ Huấn luyện CrossQ hoàn tất!')

# 4. Đánh giá trên tập Test Out-of-Sample
def crossq_policy_fn(obs, env):
    flat_obs = obs.flatten().astype(np.float32)
    action, _ = crossq_model.predict(flat_obs, deterministic=True)
    return action

m_crossq = evaluate_strategy(test_env, 'CrossQ', crossq_policy_fn)
history_crossq = test_env.asset_memory.copy()
sb3_all_results.append(m_crossq)

print('\n📊 KẾT QUẢ ĐÁNH GIÁ THỰC TẾ CrossQ:')
for k, v in m_crossq.items():
    if k != 'Strategy':
        print(f'  • {k}: {v}')

print('\n💡 NHẬN XÉT ĐẶC THÙ CHO BÀI TOÁN PORTFOLIO MANAGEMENT:')
print('  - CrossQ giải quyết bài toán trễ thông tin trong DRL tài chính: Học trực tiếp không cần chờ mạng Target bắt kịp.')
print('  - Tầng Batch Normalization giữ cho gradient ổn định bất chấp dữ liệu giá tài chính có biến động thang đo lớn.')


>>> [7/7] PHÂN TÍCH KIẾN TRÚC MẠNG & THUẬT TOÁN CrossQ (ICLR 2024)

📐 BÓC TÁCH KIẾN TRÚC MẠNG NƠ-RON CỦA CrossQ:
  • Policy Class:       CrossQPolicy
  • Cơ chế Đột phá:     HOÀN TOÀN KHÔNG SỬ DỤNG TARGET NETWORKS (No Target Delay)!
  • Batch Normalization: Áp dụng trực tiếp vào Critic: CrossQCritic(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (qf0): Sequential(
    (0): BatchRenorm1d(num_features=911, momentum=0.01, warmup_steps=100000, affine=True)
    (1): Linear(in_features=911, out_features=1024, bias=True)
    (2): ReLU()
    (3): BatchRenorm1d(num_features=1024, momentum=0.01, warmup_steps=100000, affine=True)
    (4): Linear(in_features=1024, out_features=1024, bias=True)
    (5): ReLU()
    (6): BatchRenorm1d(num_features=1024, momentum=0.01, warmup_steps=100000, affine=True)
    (7): Linear(in_features=1024, out_features=1, bias=True)
  )
  (qf1): Sequential(
    (0): BatchRenorm1d(num_features=911, momentum=0.01, warmup_ste

---
## 📊 BẢNG SO SÁNH TỔNG HỢP TOÀN BỘ 7 THUẬT TOÁN BASELINE & BENCHMARKS

Dưới đây là bảng đối chiếu hiệu suất chi tiết giữa tất cả các thuật toán Reinforcement Learning và các danh mục Benchmark chuẩn (Equal-Weight $1/N$, 100% Cash, Random):


In [17]:
# 1. Tạo DataFrame tổng hợp toàn bộ kết quả
df_final_comparison = pd.DataFrame(sb3_all_results)
df_final_comparison.set_index('Strategy', inplace=True)
print('='*85)
print('🏆 BẢNG XẾP HẠNG HIỆU SUẤT CÁC THUẬT TOÁN TRÊN TẬP TEST OUT-OF-SAMPLE:')
print('='*85)
print(df_final_comparison.to_string())

# 2. Gọi hàm plot_investment_results để trực quan hóa toàn bộ kết quả đầu tư của các mô hình
all_models_histories = {
    'Benchmark: Equal Weight (1/N)': history_ew,
    'Benchmark: Cash Only (100%)': history_cash,
    'A2C (On-Policy)': history_a2c,
    'PPO (Clipped Surrogate)': history_ppo,
    'DDPG (Deterministic)': history_ddpg,
    'TD3 (Twin Delayed)': history_td3,
    'SAC (Maximum Entropy)': history_sac,
    'TQC (Truncated Quantiles)': history_tqc,
    'CrossQ (Batch Norm DRL)': history_crossq,
}

fig_all, df_all_summary = plot_investment_results(
    histories=all_models_histories,
    df_metrics=df_final_comparison,
    title='ĐỐI CHIẾU TOÀN DIỆN KẾT QUẢ ĐẦU TƯ: 7 THUẬT TOÁN BASELINE DRL VS BENCHMARKS',
    save_path='all_models_investment_comparison.png',
    show_plot=True
)


🏆 BẢNG XẾP HẠNG HIỆU SUẤT CÁC THUẬT TOÁN TRÊN TẬP TEST OUT-OF-SAMPLE:
                    Initial Value ($)  Final Value ($)  Cumulative Return (%)  Annualized Return (%)  Annualized Volatility (%)  Sharpe Ratio  Max Drawdown (%)
Strategy                                                                                                                                                       
Equal Weight (1/N)           100000.0    113949.796875              13.949797              13.383254                   7.843622      1.451275          4.375701
Cash Only (100%)             100000.0    100000.000000               0.000000               0.000000                   0.000000      0.000000          0.000000
Random Rebalancing           100000.0     68297.992188             -31.702008             -30.700796                  10.393330     -3.146325         31.800865
A2C                          100000.0    110131.859375              10.131859               9.726933                   7.272668   